# XGBoost for Cross-Sectional Return Prediction

## Objective

Test whether allowing nonlinear relationships and feature interactions
improves next-day cross-sectional equity return prediction relative to
regularized linear models.

The linear-model benchmark from Notebook 03 showed:

- essentially zero out-of-sample point-forecast R²,
- weak but positive validation Rank IC,
- improved ranking performance under strong regularization.

XGBoost is introduced as a nonlinear challenger.

The central research question is:

> Do nonlinearities and interactions provide incremental out-of-sample predictive information beyond regularized linear models?

Model fitting uses only the training period (2010–2018).

Hyperparameters are selected using the validation period (2019–2021).

The test period (2022 onward) remains untouched.

In [1]:
import json

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from metrics import oos_r2, daily_ic, evaluate_predictions, newey_west_mean_test, newey_west_paired_test

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

df = pd.read_csv(DATA_DIR / "alpha_dataset.csv", parse_dates=["date"])

df.head()

,date,adj_close,close,high,low,open,volume,ticker,ret_1d,next_obs_date,...,beta_60,ret_1d_z,mom_5_z,mom_20_z,mom_60_z,vol_20_z,vol_60_z,price_ma20_gap_z,dollar_volume_z20_z,beta_60_z
0,2010-03-31,7.028762,8.392857,8.450357,8.373571,8.410357,430659600,AAPL,-0.003604,2010-04-01,...,1.497638,0.072730,1.577789,1.472385,0.625248,0.258562,0.577992,1.687527,-0.201478,1.199480
1,2010-03-31,39.938347,59.840000,60.220001,59.689999,60.000000,4975100,AMGN,-0.004161,2010-04-01,...,0.439355,-0.004092,0.822145,-0.028271,-0.077188,-0.257272,-0.873828,0.610112,-0.178906,-1.426402
2,2010-03-31,56.966679,72.610001,73.059998,71.949997,72.919998,5884600,BA,-0.012512,2010-04-01,...,1.289499,-1.156940,0.229550,1.556566,2.951781,0.651547,0.829979,1.036881,-0.109082,0.683032
3,2010-03-31,13.634262,17.850000,17.980000,17.670000,17.690001,114280900,BAC,0.005067,2010-04-01,...,1.586337,1.269740,1.012569,0.790991,1.086433,0.173629,1.174250,1.504354,-0.966822,1.419567
4,2010-03-31,29.604025,40.500000,41.099998,40.099998,40.799999,55035890,C,-0.009780,2010-04-01,...,1.610091,-0.779761,-1.615686,2.922020,1.691178,3.920722,2.263984,0.335038,0.354980,1.478507


In [3]:
MODEL_FEATURES = [
    "ret_1d_z",
    "mom_5_z",
    "mom_20_z",
    "mom_60_z",
    "vol_20_z",
    "vol_60_z",
    "price_ma20_gap_z",
    "dollar_volume_z20_z",
    "beta_60_z",
]

TARGET = "target_cs_1d"

In [4]:
missing_cols = set(MODEL_FEATURES + [TARGET]) - set(df.columns)

assert not missing_cols, f"Missing columns: {missing_cols}"

In [5]:
train = df[df["split"] == "train"].sort_values(["date", "ticker"]).reset_index(drop=True)

validation = df[df["split"] == "validation"].sort_values(["date", "ticker"]).reset_index(drop=True)

test = df[df["split"] == "test"].sort_values(["date", "ticker"]).reset_index(drop=True)

In [6]:
X_train = train[MODEL_FEATURES]
y_train = train[TARGET]

X_val = validation[MODEL_FEATURES]
y_val = validation[TARGET]

In [7]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test rows:", len(test))

print()
print("Missing train values:", X_train.isna().sum().sum())

print("Missing validation values:", X_val.isna().sum().sum())

Train: (66120, 9)
Validation: (22710, 9)
Test rows: 34980

Missing train values: 0
Missing validation values: 0


# XGBoost

XGBoost builds many small decision trees sequentially.

A single decision tree might learn a rule such as:

    mom_5_z < -1.2
        |
        +-- yes -> positive forecast
        |
        +-- no
              |
              vol_20_z > 0.8
                  |
                  +-- yes -> negative forecast
                  +-- no  -> approximately zero

A single tree is usually weak.

Boosting constructs a sequence:

    prediction =
        tree_1
        + learning_rate * tree_2
        + learning_rate * tree_3
        + ...

Each new tree attempts to correct errors made by the existing ensemble.

Unlike linear regression, trees can naturally represent:

- nonlinear thresholds,
- interactions between variables,
- different behavior in different regions of feature space.

The risk is overfitting, especially because next-day equity returns
contain very little signal. We therefore use shallow trees and a small
hyperparameter search.

## Define a conservative XGBoost baseline

Our problem has: $$ \text{signal} \ll \text{noise} $$
So we want to make it relatively difficult for XGBoost to memorize the training sample.

In particular, max_depth = 2 means that each tree is very simple.

In [8]:
xgb_baseline = XGBRegressor(
    objective="reg:squarederror",
    # Keep trees shallow because signal is weak
    max_depth=2,
    # Small learning rate
    learning_rate=0.05,
    # Number of boosting trees
    n_estimators=200,
    # Randomly use 80% of observations per tree
    subsample=0.8,
    # Randomly use 80% of features per tree
    colsample_bytree=0.8,
    # Minimum amount of data/effective weight
    # needed in child nodes
    min_child_weight=10,
    # L2 regularization
    reg_lambda=10.0,
    # No L1 penalty initially
    reg_alpha=0.0,
    random_state=42,
    n_jobs=-1,
)

In [9]:
xgb_baseline.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.objective.Objective, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets im

## Baseline XGBoost predictions

In [10]:
xgb_train_pred = xgb_baseline.predict(X_train)

xgb_val_pred = xgb_baseline.predict(X_val)

In [11]:
baseline_train_metrics = evaluate_predictions(
    train,
    xgb_train_pred,
    "XGBoost Baseline - Train",
)

baseline_val_metrics = evaluate_predictions(
    validation,
    xgb_val_pred,
    "XGBoost Baseline - Validation",
)

pd.DataFrame(
    [
        baseline_train_metrics,
        baseline_val_metrics,
    ]
)

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,XGBoost Baseline - Train,0.000119,0.007243,0.080617,0.061385,0.225006,0.602541
1,XGBoost Baseline - Validation,0.000226,-0.000767,0.016619,0.009866,0.250852,0.512550


## Define a small hyperparameter grid
We are only going to tune three important complexity controls to avoid overfitting the validation set

We get
$$ \text{Max Depth} \times \text{Learning Rates} \times \text{N Estimators} = 3 \times 2 \times 2 = 12 $$
models

In [12]:
MAX_DEPTHS = [1, 2, 3]

LEARNING_RATES = [0.02, 0.05]

N_ESTIMATORS = [100, 300]

In [13]:
# Hyperparameter search
xgb_results = []
xgb_models = {}

for max_depth in MAX_DEPTHS:
    for learning_rate in LEARNING_RATES:
        for n_estimators in N_ESTIMATORS:
            model = XGBRegressor(
                objective="reg:squarederror",
                max_depth=max_depth,
                learning_rate=learning_rate,
                n_estimators=n_estimators,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=10,
                reg_lambda=10.0,
                reg_alpha=0.0,
                random_state=42,
                n_jobs=-1,
            )

            # Train using only 2010-2018
            model.fit(X_train, y_train)

            # Evaluate on 2019-2021
            val_pred = model.predict(X_val)

            metrics = evaluate_predictions(
                validation,
                val_pred,
                model_name=(f"XGB depth={max_depth}, lr={learning_rate}, trees={n_estimators}"),
            )

            metrics["max_depth"] = max_depth
            metrics["learning_rate"] = learning_rate
            metrics["n_estimators"] = n_estimators

            xgb_results.append(metrics)

            xgb_models[
                (
                    max_depth,
                    learning_rate,
                    n_estimators,
                )
            ] = model

In [14]:
# Convert to dataframe
xgb_results = pd.DataFrame(xgb_results)

xgb_results = xgb_results.sort_values("mean_rank_ic", ascending=False).reset_index(drop=True)

xgb_results

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive,max_depth,learning_rate,n_estimators
0,"XGB depth=1, lr=0.02, trees=300",0.000226,0.000260,0.020207,0.015520,0.266524,0.520476,1,0.02,300
1,"XGB depth=3, lr=0.02, trees=100",0.000226,0.000093,0.020257,0.013479,0.257782,0.515192,3,0.02,100
2,"XGB depth=1, lr=0.05, trees=300",0.000226,0.000237,0.021184,0.012554,0.259998,0.520476,1,0.05,300
3,"XGB depth=2, lr=0.02, trees=300",0.000226,-0.000124,0.016779,0.011801,0.261583,0.529723,2,0.02,300
4,"XGB depth=1, lr=0.02, trees=100",0.000226,0.000196,0.018075,0.010528,0.261512,0.513871,1,0.02,100
5,"XGB depth=2, lr=0.02, trees=100",0.000226,0.000184,0.020795,0.009419,0.258808,0.507266,2,0.02,100
6,"XGB depth=2, lr=0.05, trees=100",0.000226,-0.000464,0.013836,0.008475,0.255574,0.523118,2,0.05,100
7,"XGB depth=1, lr=0.05, trees=100",0.000226,0.000228,0.018445,0.007995,0.260391,0.521797,1,0.05,100
8,"XGB depth=3, lr=0.02, trees=300",0.000226,-0.000886,0.016305,0.006607,0.248206,0.517834,3,0.02,300
9,"XGB depth=3, lr=0.05, trees=100",0.000226,-0.001053,0.013037,0.006232,0.247258,0.503303,3,0.05,100


In [15]:
# Select XGBoost using validation Rank IC
best_depth = int(xgb_results.loc[0, "max_depth"])

best_lr = float(xgb_results.loc[0, "learning_rate"])

best_n_estimators = int(xgb_results.loc[0, "n_estimators"])

In [16]:
# Print
print("Best max_depth:", best_depth)

print("Best learning_rate:", best_lr)

print("Best n_estimators:", best_n_estimators)

Best max_depth: 1
Best learning_rate: 0.02
Best n_estimators: 300


In [17]:
# Retrieve
best_xgb = xgb_models[
    (
        best_depth,
        best_lr,
        best_n_estimators,
    )
]

## XGBoost seed sensitivity

Because subsampling and column subsampling introduce randomness, the selected
hyperparameters are refit across several random seeds as a diagnostic.


In [ ]:
XGB_SEEDS = range(10)

seed_results = []

for seed in XGB_SEEDS:
    model = XGBRegressor(
        objective="reg:squarederror",
        max_depth=best_depth,
        learning_rate=best_lr,
        n_estimators=best_n_estimators,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_lambda=10.0,
        reg_alpha=0.0,
        random_state=seed,
        n_jobs=-1,
    )

    model.fit(
        X_train,
        y_train,
    )

    pred = model.predict(X_val)

    temp = validation[
        [
            "date",
            "ticker",
            TARGET,
        ]
    ].copy()

    temp["prediction"] = pred

    rank_ic = daily_ic(
        temp,
        prediction_col="prediction",
        target_col=TARGET,
        method="spearman",
    )

    seed_results.append(
        {
            "seed": seed,
            "mean_rank_ic": rank_ic.mean(),
        }
    )


seed_results = pd.DataFrame(seed_results)

seed_results

,seed,mean_rank_ic
0,0,0.012101
1,1,0.015912
2,2,0.013009
3,3,0.012187
4,4,0.013910
5,5,0.014476
6,6,0.012530
7,7,0.015884
8,8,0.011968
9,9,0.014604


In [ ]:
seed_results["mean_rank_ic"].agg(
    [
        "mean",
        "std",
        "min",
        "max",
    ]
)

mean    0.013658
std     0.001520
min     0.011968
max     0.015912
Name: mean_rank_ic, dtype: float64

## Evaluate selected XGBoost on train and validation

In [20]:
best_xgb_train_pred = best_xgb.predict(X_train)

best_xgb_val_pred = best_xgb.predict(X_val)

In [ ]:
# Metrics
best_xgb_train_metrics = evaluate_predictions(
    train,
    best_xgb_train_pred,
    "XGBoost - Train",
)

best_xgb_val_metrics = evaluate_predictions(
    validation,
    best_xgb_val_pred,
    "XGBoost - Validation",
)

xgb_train_validation = pd.DataFrame(
    [
        best_xgb_train_metrics,
        best_xgb_val_metrics,
    ]
)

# "oos_r2" is not literally out-of-sample
# for the training row.
xgb_train_validation = xgb_train_validation.rename(columns={"oos_r2": "r2_vs_zero"})

xgb_train_validation

,model,mse,r2_vs_zero,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,XGBoost - Train,0.000120,0.001437,0.037775,0.029836,0.238041,0.543103
1,XGBoost - Validation,0.000226,0.000260,0.020207,0.015520,0.266524,0.520476


`r2_vs_zero` uses the same zero-return benchmark in both rows. For the
training row it is an in-sample statistic; for the validation row it is
out-of-sample. The reduction from training to validation should therefore
be interpreted as a train-validation generalization gap rather than as
"out-of-sample decay" from the training period.

## Load final linear predictions and add XGBoost predictions

In [22]:
linear_predictions = pd.read_csv(
    DATA_DIR / "linear_model_validation_predictions.csv",
    parse_dates=["date"],
)

linear_predictions.head()

,date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred
0,2019-01-02,AAPL,-0.099608,-0.075324,-0.000052,-0.000011,-0.000081
1,2019-01-02,AMGN,-0.015216,0.009068,-0.000090,0.000003,0.000005
2,2019-01-02,BA,-0.039869,-0.015585,-0.000191,-0.000038,-0.000143
3,2019-01-02,BAC,-0.016026,0.008258,-0.000499,-0.000054,-0.000169
4,2019-01-02,C,-0.018120,0.006164,-0.000489,-0.000054,-0.000179


In [23]:
xgb_predictions = validation[
    [
        "date",
        "ticker",
        "target_cs_1d",
        "fwd_ret_1d",
    ]
].copy()

xgb_predictions["xgboost_pred"] = best_xgb_val_pred

In [24]:
# Merge
all_validation_predictions = linear_predictions.merge(
    xgb_predictions[
        [
            "date",
            "ticker",
            "xgboost_pred",
        ]
    ],
    on=[
        "date",
        "ticker",
    ],
    how="inner",
)

In [25]:
# Check
all_validation_predictions.head()

,date,ticker,fwd_ret_1d,target_cs_1d,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
0,2019-01-02,AAPL,-0.099608,-0.075324,-0.000052,-0.000011,-0.000081,-0.000055
1,2019-01-02,AMGN,-0.015216,0.009068,-0.000090,0.000003,0.000005,0.000152
2,2019-01-02,BA,-0.039869,-0.015585,-0.000191,-0.000038,-0.000143,-0.000109
3,2019-01-02,BAC,-0.016026,0.008258,-0.000499,-0.000054,-0.000169,-0.000147
4,2019-01-02,C,-0.018120,0.006164,-0.000489,-0.000054,-0.000179,-0.000248


## Direct linear vs nonlinear comparison

In [ ]:
# Verify that the merged prediction table contains exactly the same validation (date, ticker) keys.

expected_keys = validation[["date", "ticker"]].sort_values(["date", "ticker"]).reset_index(drop=True)

actual_keys = all_validation_predictions[["date", "ticker"]].sort_values(["date", "ticker"]).reset_index(drop=True)

pd.testing.assert_frame_equal(
    expected_keys,
    actual_keys,
)

assert not (all_validation_predictions.duplicated(["date", "ticker"]).any())

print("PASS: Validation prediction keys are aligned.")

PASS: Validation prediction keys are aligned.


In [ ]:
comparison = []

prediction_cols = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

for model_name, pred_col in prediction_cols.items():
    metrics = evaluate_predictions(
        all_validation_predictions,
        all_validation_predictions[pred_col].to_numpy(),
        model_name,
    )

    comparison.append(metrics)


comparison = (
    pd.DataFrame(comparison)
    .sort_values(
        "mean_rank_ic",
        ascending=False,
    )
    .reset_index(drop=True)
)

comparison

,model,mse,oos_r2,mean_pearson_ic,mean_rank_ic,rank_ic_std,pct_rank_ic_positive
0,XGBoost,0.000226,0.000260,0.020207,0.015520,0.266524,0.520476
1,Ridge,0.000226,-0.000018,0.000134,0.009320,0.310012,0.520476
2,Elastic Net,0.000226,-0.000101,-0.002767,0.008628,0.309960,0.517834
3,OLS,0.000226,-0.000175,-0.001229,0.002137,0.288068,0.513871


## Statistical uncertainty of validation Rank IC

The validation period is used for hyperparameter selection, so the following
tests are descriptive rather than confirmatory. In particular, selecting the
best configuration from a grid creates upward selection bias in the winning
validation metric.

Newey-West / HAC standard errors are used to account for possible serial
dependence in the daily Rank IC series.

In [ ]:
validation_ic_series = {}

for model_name, pred_col in prediction_cols.items():
    validation_ic_series[model_name] = daily_ic(
        all_validation_predictions,
        prediction_col=pred_col,
        target_col="target_cs_1d",
        method="spearman",
    )

In [ ]:
validation_ic_inference = []

for model_name, ic in validation_ic_series.items():
    result = newey_west_mean_test(ic)

    result["model"] = model_name

    validation_ic_inference.append(result)


validation_ic_inference = pd.DataFrame(validation_ic_inference)[
    [
        "model",
        "n",
        "mean",
        "nw_se",
        "nw_t",
        "nw_pvalue",
        "maxlags",
    ]
]

validation_ic_inference

,model,n,mean,nw_se,nw_t,nw_pvalue,maxlags
0,OLS,757,0.002137,0.010163,0.210257,0.833467,6
1,Ridge,757,0.009320,0.010818,0.861541,0.388940,6
2,Elastic Net,757,0.008628,0.010711,0.805547,0.420504,6
3,XGBoost,757,0.015520,0.010520,1.475289,0.140135,6


In [ ]:
validation_pairs = [
    (
        "Ridge",
        "OLS",
    ),
    (
        "Elastic Net",
        "OLS",
    ),
    (
        "XGBoost",
        "Ridge",
    ),
]

validation_pair_results = []

for model_a, model_b in validation_pairs:
    result = newey_west_paired_test(
        validation_ic_series[model_a],
        validation_ic_series[model_b],
    )

    result["comparison"] = f"{model_a} - {model_b}"

    validation_pair_results.append(result)


validation_pair_results = pd.DataFrame(validation_pair_results)[
    [
        "comparison",
        "n",
        "mean_a",
        "mean_b",
        "mean_difference",
        "nw_se",
        "nw_t",
        "nw_pvalue",
        "maxlags",
    ]
]

validation_pair_results

,comparison,n,mean_a,mean_b,mean_difference,nw_se,nw_t,nw_pvalue,maxlags
0,Ridge - OLS,757,0.009320,0.002137,0.007184,0.005747,1.249889,0.211340,6
1,Elastic Net - OLS,757,0.008628,0.002137,0.006491,0.005577,1.163962,0.244439,6
2,XGBoost - Ridge,757,0.015520,0.009320,0.006199,0.008577,0.722802,0.469801,6


## Feature importance

We inspect XGBoost to see which features it actually uses most.

Use gain-based importance from the underlying booster

In [31]:
booster = best_xgb.get_booster()

gain_scores = booster.get_score(importance_type="gain")

In [32]:
# Construct Table
importance = pd.DataFrame({"feature": MODEL_FEATURES})

# Map scores
importance["gain"] = importance["feature"].map(gain_scores).fillna(0)

# Normalize
importance["gain_share"] = importance["gain"] / importance["gain"].sum()

# Sort
importance = importance.sort_values("gain_share", ascending=False).reset_index(drop=True)

importance

,feature,gain,gain_share
0,dollar_volume_z20_z,0.001378,0.134491
1,ret_1d_z,0.001335,0.130340
2,vol_60_z,0.001241,0.121094
3,vol_20_z,0.001164,0.113621
4,mom_20_z,0.001159,0.113168
5,mom_5_z,0.001040,0.101546
6,price_ma20_gap_z,0.000981,0.095770
7,beta_60_z,0.000975,0.095205
8,mom_60_z,0.000971,0.094764


In [33]:
# Plot the feature importance
fig = px.bar(
    importance.sort_values("gain_share"),
    x="gain_share",
    y="feature",
    orientation="h",
    title="XGBoost Feature Importance — Gain",
)

fig.show()

A large gain value means XGBoost found the feature useful for reducing prediction error.

## Rolling Rank IC

In [34]:
# We repeat the stability diagnostic similar to notebook 03
prediction_cols = {
    "OLS": "ols_pred",
    "Ridge": "ridge_pred",
    "Elastic Net": "elastic_net_pred",
    "XGBoost": "xgboost_pred",
}

In [35]:
# Calculate daily IC
ic_frames = []

for model_name, pred_col in prediction_cols.items():
    ic = daily_ic(
        all_validation_predictions,
        prediction_col=pred_col,
        method="spearman",
    )

    temp = pd.DataFrame(
        {
            "date": ic.index,
            "rank_ic": ic.values,
            "model": model_name,
        }
    )

    ic_frames.append(temp)


rank_ic_series = pd.concat(ic_frames, ignore_index=True)

In [36]:
# Rolling 60-day mean
rank_ic_series = rank_ic_series.sort_values(["model", "date"])

rank_ic_series["rank_ic_60d"] = rank_ic_series.groupby("model")["rank_ic"].transform(
    lambda s: s.rolling(60, min_periods=20).mean()
)

In [37]:
# Plot
fig = px.line(
    rank_ic_series,
    x="date",
    y="rank_ic_60d",
    color="model",
    title=("Validation-Period 60-Day Rolling Rank IC"),
)

fig.add_hline(y=0, line_dash="dash")

fig.show()

## Prediction correlation

In [38]:
pred_corr = all_validation_predictions[
    [
        "ols_pred",
        "ridge_pred",
        "elastic_net_pred",
        "xgboost_pred",
    ]
].corr()

pred_corr

,ols_pred,ridge_pred,elastic_net_pred,xgboost_pred
ols_pred,1.000000,0.827808,0.823123,0.251934
ridge_pred,0.827808,1.000000,0.925602,0.315042
elastic_net_pred,0.823123,0.925602,1.000000,0.307020
xgboost_pred,0.251934,0.315042,0.307020,1.000000


In [39]:
fig = px.imshow(
    pred_corr,
    text_auto=".2f",
    aspect="auto",
    title=("Correlation Between Model Predictions"),
)

fig.show()

## Save XGBoost hyperparameters and combined validation predictions

In [40]:
xgb_params = {
    "max_depth": int(best_depth),
    "learning_rate": float(best_lr),
    "n_estimators": int(best_n_estimators),
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "reg_lambda": 10.0,
    "reg_alpha": 0.0,
}

In [41]:
# Save
with open(DATA_DIR / "xgboost_params.json", "w") as f:
    json.dump(
        xgb_params,
        f,
        indent=4,
    )

In [42]:
# Inspect
xgb_params

{'max_depth': 1,
 'learning_rate': 0.02,
 'n_estimators': 300,
 'subsample': 0.8,
 'colsample_bytree': 0.8,
 'min_child_weight': 10,
 'reg_lambda': 10.0,
 'reg_alpha': 0.0}

In [43]:
all_validation_predictions.to_csv(
    DATA_DIR / "model_validation_predictions.csv",
    index=False,
)

## Findings

XGBoost produced the highest observed mean validation Rank IC among the
models tested.

The validation-selected specification used depth-1 trees, a learning rate
of 0.02, and 300 boosting rounds. Its mean validation Rank IC was approximately

\[
0.0155,
\]

compared with approximately

\[
0.0093
\]

for Ridge and

\[
0.0086
\]

for Elastic Net.

However, Newey-West inference shows that none of the validation-period mean
Rank IC estimates is statistically distinguishable from zero at conventional
levels. For XGBoost,

\[
t_{NW} \approx 1.48,\qquad p \approx 0.14.
\]

Pairwise tests also do not establish statistically significant differences
between XGBoost and Ridge or between the regularized linear models and OLS.

The validation results should therefore be interpreted as model-selection
point estimates rather than evidence that one specification has established
superior predictive ability.

The selected XGBoost hyperparameters were also evaluated across multiple
random seeds. Mean validation Rank IC remained positive across all tested
seeds, ranging from approximately 0.0120 to 0.0159, indicating that the
observed validation ranking was not driven solely by a favorable random seed.
This does not eliminate sampling uncertainty or the optimism created by
selecting the best specification on the validation period.

The selected model uses depth-1 trees. This shows that complex feature
interactions were not necessary for the highest observed validation score,
but does not establish that deeper interactions are unhelpful because some
deeper specifications produced comparable results.

The final question is therefore not whether the validation winner is
statistically superior, but whether any of these frozen specifications
produce detectable and economically useful predictability in a genuinely
untouched future sample.